# Applying integrative multi-omics to the positive control case: D-gene

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

## VCF file for forward genetics
Opening and analysing the variant file containing variant sites exclusive to SBC4, 10, 23, samples that show "juiciness" phenotype.

In [21]:
%%bash
VCF_PATH=../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
../docker/run.sh python3 ../workflow/scripts/genomics_scoring.py -i $VCF_PATH -o ../results/gene_score/ -g ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz

[genomics_scoring] Parsing VCF: ../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
[genomics_scoring]   205,996 (variant, gene) rows
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant.tsv  (205,996 variant annotations)
[genomics_scoring] Loading gene lengths: ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz
[genomics_scoring]   32,458 genes with length
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.scores_summary.png
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant_scores.png


### Choosing the most appropriate merging method out of the four
Cost function: which method places D-gene on the highest rank?

In [22]:
df_scored               = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.variant.tsv", sep="\t")
df_scored_gene_max      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv", sep="\t")
df_scored_gene_sum      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv", sep="\t") 
df_scored_gene_max_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv", sep="\t")
df_scored_gene_sum_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv", sep="\t")

In [23]:
# Confirmed dPCD homologs in sorghum (Olvera-Carrillo 2015; ATTED-II orthologs).

def where_genes(df, genes):
    """Rank & percentile of a gene set within a gene-score table.

    df    : gene-level score table keyed by `ann_gene_id` (= "LOC<EGI>"),
            e.g. df_scored_gene_max; must carry `rank`, `percentile`.
    genes : {ann_gene_id: label}. A gene absent from df gets a NaN row
            (= it carries no variant in this sharing group).
    Returns one row per gene in `genes`, sorted by rank (found genes first).
    """
    cols = [c for c in ["score", "percentile", "rank"] if c in df.columns]
    out = (df[df["ann_gene_id"].isin(genes)]
           .drop_duplicates("ann_gene_id")
           .set_index("ann_gene_id")[cols]
           .reindex(genes.keys()))
    out.insert(0, "gene", out.index.map(genes))
    out.insert(1, "found", out["rank"].notna())
    return out.sort_values("rank", na_position="last")

In [24]:
# Gene-merging methods paired with readable labels, for iterating over all four
merged_dfs = {
    "gene_max":      df_scored_gene_max,
    "gene_sum":      df_scored_gene_sum,
    "gene_max_norm": df_scored_gene_max_norm,
    "gene_sum_norm": df_scored_gene_sum_norm,
}

## DMR analysis for forward genetics

In [25]:
from pathlib import Path

# DMR_annotated.tsv schema (one row per DMR x overlapping feature x gene):
#   chr, start, end, diff.Methy, direction (= "hyper_<accession>"),
#   sample_a, sample_b, feature (promoter|exon|CDS|intron|intergenic), gene_label
DMR_PATH = Path("../results/DMR/DMR_annotated.tsv")

df_dmr = pd.read_csv(DMR_PATH, sep="\t")
print(f"{len(df_dmr):,} DMR-feature rows | features: {sorted(df_dmr['feature'].unique())}")

# --- Candidate genes: promoter HYPOMETHYLATED in the dry accession SBC11 -------
# SBC11 is the lone dry accession. A promoter LESS methylated in SBC11 than in a
# juicy accession is de-repressed in SBC11 / silenced in the juicy ones -- the
# expression contrast that could underlie the juicy-vs-dry split. So candidates
# are genes whose promoter is hypomethylated in SBC11.
#
# Direction encoding: `direction` names the HYPER accession ("hyper_<acc>").
# Hypomethylated in SBC11  <=>  the contrast involves SBC11 AND the hyper side is
# the OTHER accession, i.e. direction != "hyper_SBC11".
SBC11 = "SBC11"

promoter_dmr = df_dmr[
    (df_dmr["feature"] == "promoter")
    & df_dmr["gene_label"].notna()
    & (df_dmr["gene_label"].astype(str).str.strip() != "")
].copy()

involves_sbc11 = (promoter_dmr["sample_a"] == SBC11) | (promoter_dmr["sample_b"] == SBC11)
hypo_in_sbc11  = involves_sbc11 & (promoter_dmr["direction"] != f"hyper_{SBC11}")
sbc11_hypo = promoter_dmr[hypo_in_sbc11].copy()
sbc11_hypo["abs_diff"] = sbc11_hypo["diff.Methy"].abs()

# one row per gene: its strongest SBC11-hypomethylated promoter DMR
promoter_candidates = (
    sbc11_hypo
    .sort_values("abs_diff", ascending=False)
    .drop_duplicates(subset="gene_label")
    .loc[:, ["gene_label", "diff.Methy", "abs_diff", "direction",
             "sample_a", "sample_b", "chr", "start", "end"]]
    .reset_index(drop=True)
)
print(f"{int(hypo_in_sbc11.sum()):,} promoter DMRs hypomethylated in {SBC11} -> "
      f"{len(promoter_candidates):,} candidate genes")
promoter_candidates.head(20)

152,253 DMR-feature rows | features: ['CDS', 'exon', 'intergenic', 'promoter']
6,961 promoter DMRs hypomethylated in SBC11 -> 5,367 candidate genes


,gene_label,diff.Methy,abs_diff,direction,sample_a,sample_b,chr,start,end
0,LOC110432410,-0.621911,0.621911,hyper_SBC23,SBC11,SBC23,NC_012871.2,9293745,9293880
1,LOC110432674,-0.618042,0.618042,hyper_SBC4,SBC11,SBC4,NC_012871.2,36039726,36039979
2,LOC110436852,-0.569947,0.569947,hyper_SBC4,SBC11,SBC4,NC_012876.2,63484731,63485968
3,LOC110437322,-0.569947,0.569947,hyper_SBC4,SBC11,SBC4,NC_012876.2,63484731,63485968
4,LOC110433273,-0.562994,0.562994,hyper_SBC23,SBC11,SBC23,NC_012872.2,17366045,17366207
5,LOC8074345,-0.554586,0.554586,hyper_SBC23,SBC11,SBC23,NC_012874.2,65779128,65779445
6,LOC8079764,-0.540058,0.540058,hyper_SBC23,SBC11,SBC23,NC_012870.2,22449132,22449709
7,LOC8064491,-0.533174,0.533174,hyper_SBC23,SBC11,SBC23,NC_012870.2,13890142,13890569
8,LOC8062471,-0.530691,0.530691,hyper_SBC23,SBC11,SBC23,NC_012870.2,13225228,13227500
9,LOC8077906,-0.512978,0.512978,hyper_SBC23,SBC11,SBC23,NC_012878.2,45590629,45590874


### Promoter hypomethylated in SBC11 across the dry-vs-juicy contrasts

In [26]:
# --- Refinement: hypomethylated in SBC11 CONSISTENTLY across dry-vs-juicy ------
# Strongest candidates are hypomethylated in SBC11 in EVERY SBC11-vs-juicy
# contrast where the gene's promoter appears -- not just one -- so the signal
# tracks the 3-vs-1 phenotype split rather than a single accession pair.
JUICY = ["SBC4", "SBC10", "SBC23"]

dry_vs_juicy = promoter_dmr[
    ((promoter_dmr["sample_a"] == SBC11) & promoter_dmr["sample_b"].isin(JUICY))
    | ((promoter_dmr["sample_b"] == SBC11) & promoter_dmr["sample_a"].isin(JUICY))
].copy()
dry_vs_juicy["hypo_in_dry"] = dry_vs_juicy["direction"] != f"hyper_{SBC11}"

consistency = (
    dry_vs_juicy.groupby("gene_label")
    .agg(n_contrasts=("direction", "size"),
         n_hypo_dry=("hypo_in_dry", "sum"),
         mean_diff=("diff.Methy", "mean"))
)
# keep genes hypomethylated in SBC11 in every dry-vs-juicy contrast they appear in
consistency["all_hypo_dry"] = consistency["n_hypo_dry"] == consistency["n_contrasts"]

phenotype_candidates = (
    consistency[consistency["all_hypo_dry"] & (consistency["n_contrasts"] >= 2)]
    .sort_values("mean_diff", key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
print(f"{len(phenotype_candidates)} genes with a promoter hypomethylated in {SBC11} "
      f"across >=2 dry-vs-juicy contrasts")
phenotype_candidates.head(20)

1204 genes with a promoter hypomethylated in SBC11 across >=2 dry-vs-juicy contrasts


,gene_label,n_contrasts,n_hypo_dry,mean_diff,all_hypo_dry
0,LOC8080241,2,2,-0.442617,True
1,LOC8079764,2,2,-0.414391,True
2,LOC8062471,2,2,-0.371470,True
3,LOC8084237,2,2,-0.356687,True
4,LOC110431956,2,2,-0.330738,True
5,LOC110432782,2,2,-0.330057,True
6,LOC8068093,2,2,0.297456,True
7,LOC8072946,2,2,-0.283170,True
8,LOC8080209,2,2,-0.282329,True
9,LOC8084307,2,2,0.274740,True


In [27]:
ROOT    = Path("/Users/daffa/workspace/infobio")
ZD      = ROOT / "Sbi-r.c1-0" / "Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d"
TF_NAC  = ROOT / "thesis" / "resources" / "TFDB" / "Sbi_TF_list_NAC.txt"

## dPCD gene module

### dPCD gene module from literature review

| AGI | Name | Homologs in sorghum (EGI) | Name in sorghum|
|-----|------|---------------------------|----------------|
| AT1G26820 | ribonuclease 3 | LOC8063515 <br> LOC8063317 <br> LOC8077825 | ribonuclease 1 <br> ribonuclease 3 <br> ribonuclease 1 |
| AT1G11190 | bifunctional nuclease i | - | - | 
| AT5G04200 | metacaspase 9 | - | - |
| AT3G45010 | serine carboxypeptidase-like 48 | LOC8074232 | serine carboxypeptidase-like |
| AT1G62290 | Saposin-like aspartyl protease family protein | LOC8058198 <br> LOC110430277 <br> LOC110433911 | aspartic proteinase oryzasin-1 <br> aspartic proteinase-like <br> aspartic proteinase-like |

8063515
8063317
8077825
8074232
8058198
110430277
110433911

### Finding these dPCD gene module in the variant & DMR genes lists

In [28]:
# dPCD module: sorghum homologs from the literature-review table above
# (CEP1/SEN102 dropped). Keyed by ann_gene_id (= "LOC<EGI>").
MODULE = {
    "LOC8063515":  "ribonuclease 1 (RNS3-a)",
    "LOC8063317":  "ribonuclease 3 (RNS3-b)",
    "LOC8077825":  "ribonuclease 1 (RNS3-c)",
    "LOC8074232":  "serine carboxypeptidase-like (SCPL48)",
    "LOC8058198":  "aspartic proteinase oryzasin-1 (PASPA3-a)",
    "LOC110430277": "aspartic proteinase-like (PASPA3-b)",
    "LOC110433911": "aspartic proteinase-like (PASPA3-c)",
}

# 1) Variant evidence -- rank of the module under each gene-merging method
print("=== Variant gene-score rank of the dPCD module (SBC4_SBC10_SBC23 group) ===")
for name, df in merged_dfs.items():
    print(f"\n[{name}]")
    display(where_genes(df, MODULE))

# 2) DMR evidence -- is any module gene a SBC11-hypomethylated promoter candidate?
print("\n=== DMR promoter evidence (SBC11-hypomethylated candidates) ===")
dmr_hit = promoter_candidates[promoter_candidates["gene_label"].isin(MODULE)].copy()
if dmr_hit.empty:
    print("no module gene among SBC11-hypomethylated promoter candidates")
else:
    dmr_hit.insert(0, "gene", dmr_hit["gene_label"].map(MODULE))
    display(dmr_hit)

=== Variant gene-score rank of the dPCD module (SBC4_SBC10_SBC23 group) ===

[gene_max]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8063515,ribonuclease 1 (RNS3-a),True,0.125,45.064643,92.0
LOC8063317,ribonuclease 3 (RNS3-b),False,NaN,NaN,NaN
LOC8077825,ribonuclease 1 (RNS3-c),False,NaN,NaN,NaN
LOC8074232,serine carboxypeptidase-like (SCPL48),False,NaN,NaN,NaN
LOC8058198,aspartic proteinase oryzasin-1 (PASPA3-a),False,NaN,NaN,NaN
LOC110430277,aspartic proteinase-like (PASPA3-b),False,NaN,NaN,NaN
LOC110433911,aspartic proteinase-like (PASPA3-c),False,NaN,NaN,NaN



[gene_sum]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8063515,ribonuclease 1 (RNS3-a),True,0.125,15.456375,786.0
LOC8063317,ribonuclease 3 (RNS3-b),False,NaN,NaN,NaN
LOC8077825,ribonuclease 1 (RNS3-c),False,NaN,NaN,NaN
LOC8074232,serine carboxypeptidase-like (SCPL48),False,NaN,NaN,NaN
LOC8058198,aspartic proteinase oryzasin-1 (PASPA3-a),False,NaN,NaN,NaN
LOC110430277,aspartic proteinase-like (PASPA3-b),False,NaN,NaN,NaN
LOC110433911,aspartic proteinase-like (PASPA3-c),False,NaN,NaN,NaN



[gene_max_norm]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8063515,ribonuclease 1 (RNS3-a),True,0.10322,74.50642,1412.0
LOC8063317,ribonuclease 3 (RNS3-b),False,NaN,NaN,NaN
LOC8077825,ribonuclease 1 (RNS3-c),False,NaN,NaN,NaN
LOC8074232,serine carboxypeptidase-like (SCPL48),False,NaN,NaN,NaN
LOC8058198,aspartic proteinase oryzasin-1 (PASPA3-a),False,NaN,NaN,NaN
LOC110430277,aspartic proteinase-like (PASPA3-b),False,NaN,NaN,NaN
LOC110433911,aspartic proteinase-like (PASPA3-c),False,NaN,NaN,NaN



[gene_sum_norm]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8063515,ribonuclease 1 (RNS3-a),True,0.10322,40.480464,3952.0
LOC8063317,ribonuclease 3 (RNS3-b),False,NaN,NaN,NaN
LOC8077825,ribonuclease 1 (RNS3-c),False,NaN,NaN,NaN
LOC8074232,serine carboxypeptidase-like (SCPL48),False,NaN,NaN,NaN
LOC8058198,aspartic proteinase oryzasin-1 (PASPA3-a),False,NaN,NaN,NaN
LOC110430277,aspartic proteinase-like (PASPA3-b),False,NaN,NaN,NaN
LOC110433911,aspartic proteinase-like (PASPA3-c),False,NaN,NaN,NaN



=== DMR promoter evidence (SBC11-hypomethylated candidates) ===


,gene,gene_label,diff.Methy,abs_diff,direction,sample_a,sample_b,chr,start,end
136,aspartic proteinase-like (PASPA3-b),LOC110430277,-0.323093,0.323093,hyper_SBC4,SBC11,SBC4,NC_012878.2,3172974,3173587


# Conclusion
- Based on the genomics data (variant analysis), D-gene, the transcription factor that causes the death of pith parenchyma cells hence the causative for dry stem phenotype, is not found to be mutated/having LoF in the juicy stem samples' genome
- Based on the epigenomics data (DMR analysis), D-gene was also not found to be repressed in juicy stem samples' methylome
- It is inferred that the juicy phenotype observed is caused by mutation in the dPCD genes, that are genes upregulated by the D-gene transcription factor
- We obtained a list of dPCD genes in model species from literature review, finding the sorghum homologs, and improving its 'purity' by referring to the gene co-expression data in sorghum
- Going back to the sample data, none of the dPCD module genes (LOC8077825, LOC8058198) is found to contain a variant in the juicy trio (SBC4, SBC10, SBC23) variant group
- Meanwhile, Epigenomics (DMR analysis) data also doesn't have information on these dPCD genes (none appears among the SBC11-hypomethylated promoter candidates)